# Traffic-Based Next-Road Prediction

## Research Objective

This project investigates whether historical traffic patterns and recent
vehicle trajectory information can improve next-road prediction in an
urban road network.

Given a vehicle's incoming road, current intersection, and time of day,
the task is to estimate the probability of each valid outgoing road.

We compare a historical turning-probability baseline with a LightGBM
model that incorporates temporal traffic patterns and the vehicle's
two preceding roads.

Model performance is evaluated using multiclass log loss on a
trip-separated validation set and an independent held-out test set.
A trip-level bootstrap is used to quantify uncertainty in the
relative improvement over the baseline.

### Import and project setup

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the project root
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Make the src/ package importable
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ev_traffic.trajectories import parse_trajectory, extract_movements
from ev_traffic.network import (
    find_highway_exit_candidates,
    classify_highway_movement,
)
from ev_traffic.baselines import (
    build_reference_baseline,
    build_reference_hourly,
)
from ev_traffic.features import (
    build_candidates,
    attach_hourly_features,
    build_trajectory_history,
    attach_history_features,
)
from ev_traffic.modeling import MODEL_FEATURES, train_lightgbm
from ev_traffic.evaluation import (
    normalize_candidate_scores,
    calculate_log_loss,
    bootstrap_trip_improvement,
)

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "shenzhen"

print("Project root:", PROJECT_ROOT)
print("Data directory exists:", DATA_DIR.exists())
print("All imports successful.")

Project root: C:\Users\EAG28\OneDrive\Документы\2026W\Project\ev-traffic-charging-prediction
Data directory exists: True
All imports successful.


### Load the dataset

In [2]:
# Load the Shenzhen road network and vehicle trajectories
edges = pd.read_csv(DATA_DIR / "edge_shenzhen.csv")
nodes = pd.read_csv(DATA_DIR / "node_shenzhen.csv")
trajectories = pd.read_csv(DATA_DIR / "traj_shenzhen_20201104.csv")

print("Road edges:", len(edges))
print("Road nodes:", len(nodes))
print("Vehicle trajectory records:", len(trajectories))

print("\nEdge columns:", edges.columns.tolist())
print("Node columns:", nodes.columns.tolist())
print("Trajectory columns:", trajectories.columns.tolist())

Road edges: 27410
Road nodes: 11933
Vehicle trajectory records: 569057

Edge columns: ['Origin', 'Destination', 'Class', 'Geometry', 'Length']
Node columns: ['NodeID', 'Longitude', 'Latitude', 'HasCamera']
Trajectory columns: ['VehicleID', 'TripID', 'Points', 'DepartureTime', 'Duration', 'Length']


### Road network

In [3]:
# Construct valid turns from consecutive directed road edges
incoming_edges = edges[
    ["Origin", "Destination"]
].rename(
    columns={
        "Origin": "incoming",
        "Destination": "intersection",
    }
)

outgoing_edges = edges[
    ["Origin", "Destination"]
].rename(
    columns={
        "Origin": "intersection",
        "Destination": "outgoing",
    }
)

turn_options = (
    incoming_edges
    .merge(outgoing_edges, on="intersection", how="inner")
    [["incoming", "intersection", "outgoing"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Valid turn options:", len(turn_options))
print("Unique intersections:", turn_options["intersection"].nunique())
print("Duplicate turns:", turn_options.duplicated().sum())

display(turn_options.head())

Valid turn options: 72495
Unique intersections: 11840
Duplicate turns: 0


,incoming,intersection,outgoing
0,0,1,10107
1,0,2,8
2,0,2,7
3,1,10107,3903
4,2,8,11780


### Movements from vehicle trajectories

In [4]:
# Extract road movements from complete vehicle trajectories
movement_records = []

for row in trajectories.itertuples(index=False):
    points = parse_trajectory(row.Points)

    for incoming, intersection, outgoing, timestamp in extract_movements(points):
        movement_records.append(
            (
                row.VehicleID,
                row.TripID,
                incoming,
                intersection,
                outgoing,
                timestamp,
            )
        )

movements = pd.DataFrame(
    movement_records,
    columns=[
        "VehicleID",
        "TripID",
        "incoming",
        "intersection",
        "outgoing",
        "timestamp",
    ],
)

print("Extracted movements:", len(movements))
print("Unique trips:", movements[["VehicleID", "TripID"]].drop_duplicates().shape[0])
print("Missing values:", movements.isna().sum().sum())

display(movements.head())

Extracted movements: 14416328
Unique trips: 553825
Missing values: 0


,VehicleID,TripID,incoming,intersection,outgoing,timestamp
0,0,0,1226,11706,1227,41546.2
1,0,0,11706,1227,10226,41552.5
2,0,0,1227,10226,11320,41560.1
3,0,0,10226,11320,11699,41574.3
4,0,0,11320,11699,1188,41608.0


### Modeling dataset

In [5]:
# Derive the hour from the trajectory timestamp
movements["hour"] = (
    movements["timestamp"] // 3600
).astype("int16")

# Restrict the analysis to the original observation window
model_movements = movements.loc[
    movements["hour"].between(9, 18)
].copy()

# Derive minute of day
model_movements["minute_of_day"] = (
    model_movements["timestamp"] // 60
).astype("int16")

print("Modeling movements:", len(model_movements))
print("Excluded movements:", len(movements) - len(model_movements))
print("Hours:", sorted(model_movements["hour"].unique()))
print(
    "Timestamp range:",
    model_movements["timestamp"].min(),
    model_movements["timestamp"].max(),
)

Modeling movements: 14313159
Excluded movements: 103169
Hours: [np.int16(9), np.int16(10), np.int16(11), np.int16(12), np.int16(13), np.int16(14), np.int16(15), np.int16(16), np.int16(17), np.int16(18)]
Timestamp range: 32400.0 68399.9


### Trip-level train/validation/test split

In [6]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# Identify unique vehicle trips
unique_trips = (
    model_movements[["VehicleID", "TripID"]]
    .drop_duplicates()
)

# Split complete trips, not individual movements
train_trips, remaining_trips = train_test_split(
    unique_trips,
    test_size=0.30,
    random_state=RANDOM_STATE,
)

val_trips, test_trips = train_test_split(
    remaining_trips,
    test_size=0.50,
    random_state=RANDOM_STATE,
)

# Assign each trip to one split
trip_splits = unique_trips.copy()
trip_splits["split"] = "test"
trip_splits.loc[train_trips.index, "split"] = "train"
trip_splits.loc[val_trips.index, "split"] = "validation"

# Attach split labels to movements
model_movements = model_movements.merge(
    trip_splits,
    on=["VehicleID", "TripID"],
    how="left",
    validate="many_to_one",
)

print("Trips per split:")
print(trip_splits["split"].value_counts())

print("\nMovements per split:")
print(model_movements["split"].value_counts())

print("\nMissing split labels:", model_movements["split"].isna().sum())

Trips per split:
split
train         382671
test           82001
validation     82001
Name: count, dtype: int64

Movements per split:
split
train         10050524
test           2167081
validation     2095554
Name: count, dtype: int64

Missing split labels: 0


### Separating the datasets

In [7]:
train_movements = model_movements.loc[
    model_movements["split"] == "train"
].copy()

validation_movements = model_movements.loc[
    model_movements["split"] == "validation"
].copy()

test_movements = model_movements.loc[
    model_movements["split"] == "test"
].copy()

# Verify that no trip appears in multiple datasets
train_trip_ids = set(zip(train_movements["VehicleID"], train_movements["TripID"]))
val_trip_ids = set(zip(validation_movements["VehicleID"], validation_movements["TripID"]))
test_trip_ids = set(zip(test_movements["VehicleID"], test_movements["TripID"]))

assert train_trip_ids.isdisjoint(val_trip_ids)
assert train_trip_ids.isdisjoint(test_trip_ids)
assert val_trip_ids.isdisjoint(test_trip_ids)

print("Training movements:", len(train_movements))
print("Validation movements:", len(validation_movements))
print("Test movements:", len(test_movements))
print("Trip overlap: None")

Training movements: 10050524
Validation movements: 2095554
Test movements: 2167081
Trip overlap: None


### Reference/model-training split

In [8]:
# Identify unique training trips
unique_train_trips = (
    train_movements[["VehicleID", "TripID"]]
    .drop_duplicates()
)

# Split complete trips into reference (70%) and model-training (30%)
reference_trips, model_trips = train_test_split(
    unique_train_trips,
    test_size=0.30,
    random_state=RANDOM_STATE,
)

# Assign movements to their corresponding subsets
reference_movements = train_movements.merge(
    reference_trips,
    on=["VehicleID", "TripID"],
    how="inner",
)

model_train_movements = train_movements.merge(
    model_trips,
    on=["VehicleID", "TripID"],
    how="inner",
)

# Verify the split
assert (
    len(reference_movements) + len(model_train_movements)
    == len(train_movements)
)

reference_trip_ids = set(
    zip(reference_trips["VehicleID"], reference_trips["TripID"])
)
model_trip_ids = set(
    zip(model_trips["VehicleID"], model_trips["TripID"])
)

assert reference_trip_ids.isdisjoint(model_trip_ids)

print("Historical reference movements:", len(reference_movements))
print("Model training movements:", len(model_train_movements))
print("Total:", len(reference_movements) + len(model_train_movements))
print("Trip overlap: None")

Historical reference movements: 7015331
Model training movements: 3035193
Total: 10050524
Trip overlap: None


### Historical turning-probability baseline

In [9]:
# Build the historical turning-probability baseline
reference_baseline = build_reference_baseline(
    reference_movements,
    turn_options,
    alpha=1.0,
)

# Validate the baseline
probability_sums = (
    reference_baseline
    .groupby(["incoming", "intersection"])["historical_probability"]
    .sum()
)

print("Baseline rows:", len(reference_baseline))
print(
    "Missing historical features:",
    reference_baseline[
        [
            "historical_count",
            "historical_total",
            "historical_probability",
            "K",
        ]
    ].isna().sum().sum(),
)
print(
    "Probabilities sum to 1:",
    np.allclose(probability_sums, 1.0),
)

display(reference_baseline.head())

Baseline rows: 72495
Missing historical features: 0
Probabilities sum to 1: True


,incoming,intersection,outgoing,historical_count,K,historical_total,historical_probability
0,0,1,10107,0.0,1,0.0,1.0
1,0,2,8,0.0,2,0.0,0.5
2,0,2,7,0.0,2,0.0,0.5
3,1,10107,3903,0.0,1,0.0,1.0
4,2,8,11780,0.0,1,0.0,1.0


### Hourly and adaptive baselines

In [10]:
# Build hourly turning probabilities from reference movements
reference_hourly = build_reference_hourly(
    reference_movements=reference_movements,
    reference_baseline=reference_baseline,
    hours=model_train_movements["hour"].unique(),
    alpha=1.0,
    lambda_value=100.0,
)

# Validate both probability distributions
probability_sums = (
    reference_hourly
    .groupby(["incoming", "intersection", "hour"])[
        ["hourly_probability", "adaptive_probability"]
    ]
    .sum()
)

print("Hourly baseline rows:", len(reference_hourly))
print("Missing values:", reference_hourly.isna().sum().sum())

print(
    "Hourly probabilities sum to 1:",
    np.allclose(probability_sums["hourly_probability"], 1.0),
)

print(
    "Adaptive probabilities sum to 1:",
    np.allclose(probability_sums["adaptive_probability"], 1.0),
)

display(reference_hourly.head())

Hourly baseline rows: 724950
Missing values: 0
Hourly probabilities sum to 1: True
Adaptive probabilities sum to 1: True


,incoming,intersection,outgoing,historical_probability,K,hour,hourly_count,hourly_total,hourly_probability,adaptive_probability
0,0,1,10107,1.0,1,9,0.0,0.0,1.0,1.0
1,0,1,10107,1.0,1,10,0.0,0.0,1.0,1.0
2,0,1,10107,1.0,1,11,0.0,0.0,1.0,1.0
3,0,1,10107,1.0,1,12,0.0,0.0,1.0,1.0
4,0,1,10107,1.0,1,13,0.0,0.0,1.0,1.0


## Highway Network and Exit Analysis

In addition to general next-road prediction, this project investigates
vehicle decisions at highway exits.

We identify potential highway exit locations using the road network's
classification information. These locations will later be used to
evaluate how well the prediction model distinguishes between continuing
on a highway and taking an exit.

This analysis does not modify the model's training features.

In [11]:
# Identify potential highway exit locations
highway_exit_candidates = find_highway_exit_candidates(edges)

print(
    "Potential highway exit locations:",
    len(highway_exit_candidates),
)

print(
    "Example intersection IDs:",
    sorted(highway_exit_candidates)[:10],
)

Potential highway exit locations: 196
Example intersection IDs: [0, 3, 17, 23, 25, 33, 36, 50, 53, 56]


### Classifying observed highway movements

In [12]:
# Create a lookup for road classifications
road_classes = edges[
    ["Origin", "Destination", "Class"]
].drop_duplicates(
    subset=["Origin", "Destination"]
)

incoming_classes = road_classes.rename(
    columns={
        "Origin": "incoming",
        "Destination": "intersection",
        "Class": "incoming_class",
    }
)

outgoing_classes = road_classes.rename(
    columns={
        "Origin": "intersection",
        "Destination": "outgoing",
        "Class": "outgoing_class",
    }
)

# Select movements at potential highway exits
highway_movements = model_movements.loc[
    model_movements["intersection"].isin(highway_exit_candidates)
].copy()

# Attach road classifications
highway_movements = highway_movements.merge(
    incoming_classes,
    on=["incoming", "intersection"],
    how="left",
    validate="many_to_one",
)

highway_movements = highway_movements.merge(
    outgoing_classes,
    on=["intersection", "outgoing"],
    how="left",
    validate="many_to_one",
)

# Classify each movement
highway_movements["highway_action"] = [
    classify_highway_movement(
        incoming_class,
        outgoing_class,
        intersection,
        highway_exit_candidates,
    )
    for incoming_class, outgoing_class, intersection in zip(
        highway_movements["incoming_class"],
        highway_movements["outgoing_class"],
        highway_movements["intersection"],
    )
]

print("Movements at potential highway exits:", len(highway_movements))

print("\nHighway actions:")
print(highway_movements["highway_action"].value_counts(dropna=False))

print("\nActions by dataset split:")
print(
    pd.crosstab(
        highway_movements["split"],
        highway_movements["highway_action"],
    )
)

Movements at potential highway exits: 39282

Highway actions:
highway_action
exit        22057
continue    13666
NaN          3559
Name: count, dtype: int64

Actions by dataset split:
highway_action  continue   exit
split                          
test                1941   3300
train               9597  15441
validation          2128   3316


### Highway exit coverage

In [13]:
classified_highway = highway_movements.loc[
    highway_movements["highway_action"].isin(["continue", "exit"])
].copy()

observed_exits = classified_highway.loc[
    classified_highway["highway_action"] == "exit",
    "intersection",
].nunique()

observed_locations = classified_highway["intersection"].nunique()

print("Potential highway exit locations:", len(highway_exit_candidates))
print("Locations with classified movements:", observed_locations)
print("Locations with observed exit movements:", observed_exits)

print("\nClassified movements by split:")
print(
    pd.crosstab(
        classified_highway["split"],
        classified_highway["highway_action"],
        margins=True,
    )
)

Potential highway exit locations: 196
Locations with classified movements: 14
Locations with observed exit movements: 9

Classified movements by split:
highway_action  continue   exit    All
split                                 
test                1941   3300   5241
train               9597  15441  25038
validation          2128   3316   5444
All                13666  22057  35723


### LightGBM training candidates

In [14]:
# Generate candidate outgoing roads for training movements
model_train_candidates = build_candidates(
    model_train_movements,
    turn_options,
    reference_baseline,
)

# Attach hourly and adaptive probability features
model_train_candidates = attach_hourly_features(
    model_train_candidates,
    reference_hourly,
)

print("Training movements:", len(model_train_movements))
print("Candidate rows:", len(model_train_candidates))
print(
    "Unique movements:",
    model_train_candidates["movement_id"].nunique(),
)

print(
    "Exactly one chosen candidate per movement:",
    model_train_candidates.groupby("movement_id")["chosen"]
    .sum().eq(1).all(),
)

print(
    "Missing hourly features:",
    model_train_candidates[
        [
            "hourly_count",
            "hourly_total",
            "hourly_probability",
            "adaptive_probability",
        ]
    ].isna().sum().sum(),
)

Training movements: 3035193
Candidate rows: 7980602
Unique movements: 3035193
Exactly one chosen candidate per movement: True
Missing hourly features: 0


### Trajectory-history features

In [15]:
# Build trajectory history for training movements
model_train_history = build_trajectory_history(
    model_train_movements
)

# Match history rows to candidate movement IDs
model_train_history["movement_id"] = np.arange(
    len(model_train_history),
    dtype=np.int32,
)

# Attach trajectory-history features to candidates
model_train_candidates = attach_history_features(
    model_train_candidates,
    model_train_history,
)

print("Candidate rows:", len(model_train_candidates))

print(
    "Missing trajectory-history features:",
    model_train_candidates[
        ["previous_road_1", "previous_road_2", "trip_position"]
    ].isna().sum().sum(),
)

print(
    "Movements at the beginning of a trip:",
    (model_train_history["trip_position"] == 0).sum(),
)

display(
    model_train_candidates[
        [
            "movement_id",
            "incoming",
            "intersection",
            "outgoing",
            "chosen",
            "previous_road_1",
            "previous_road_2",
            "trip_position",
        ]
    ].head(10)
)

Candidate rows: 7980602
Missing trajectory-history features: 0
Movements at the beginning of a trip: 114802


,movement_id,incoming,intersection,outgoing,chosen,previous_road_1,previous_road_2,trip_position
0,0,1226,11706,1227,1,-1,-1,0
1,0,1226,11706,1226,0,-1,-1,0
2,1,11706,1227,10226,1,1226,-1,1
3,2,1227,10226,11320,1,11706,1226,2
4,2,1227,10226,11331,0,11706,1226,2
5,2,1227,10226,1234,0,11706,1226,2
6,3,10226,11320,11318,0,1227,11706,3
7,3,10226,11320,11699,1,1227,11706,3
8,4,11320,11699,1495,0,10226,1227,4
9,4,11320,11699,1188,1,10226,1227,4


### Train the LightGBM model

In [16]:
# Train the final LightGBM model
lightgbm_model = train_lightgbm(model_train_candidates)

print("LightGBM training complete.")
print("Number of features:", lightgbm_model.n_features_in_)
print("Number of boosting iterations:", lightgbm_model.n_estimators)

# Inspect feature importance
feature_importance = (
    pd.DataFrame({
        "feature": MODEL_FEATURES,
        "importance": lightgbm_model.feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(feature_importance)

LightGBM training complete.
Number of features: 14
Number of boosting iterations: 200


,feature,importance
0,previous_road_2,1560
1,outgoing,1106
2,previous_road_1,842
3,incoming,482
4,adaptive_probability,437
5,historical_probability,368
6,intersection,327
7,hourly_probability,221
8,hourly_total,150
9,historical_total,147


### Validation sample

In [17]:
# Calculate trajectory history on the complete validation set
validation_history = build_trajectory_history(
    validation_movements
)

# Sample validation movements while preserving original indices
validation_sample = validation_movements.sample(
    n=10_000,
    random_state=RANDOM_STATE,
)

# Retrieve the corresponding trajectory-history features
validation_sample_history = validation_history.loc[
    validation_sample.index,
    ["previous_road_1", "previous_road_2", "trip_position"],
].reset_index(drop=True)

# Reset indices and assign movement IDs
validation_sample = validation_sample.reset_index(drop=True)

validation_sample_history["movement_id"] = np.arange(
    len(validation_sample),
    dtype=np.int32,
)

print("Validation movements:", len(validation_sample))
print("History rows:", len(validation_sample_history))

print(
    "Missing history values:",
    validation_sample_history.isna().sum().sum(),
)

print(
    "Validation trips:",
    validation_sample[["VehicleID", "TripID"]]
    .drop_duplicates().shape[0],
)

display(validation_sample_history.head())

Validation movements: 10000
History rows: 10000
Missing history values: 0
Validation trips: 7919


,previous_road_1,previous_road_2,trip_position,movement_id
0,6892,6891,46,0
1,11861,11860,7,1
2,8453,8452,24,2
3,11700,4190,3,3
4,6755,11867,423,4


### Validaiton candidates

In [18]:
# Generate outgoing-road candidates for validation
validation_candidates = build_candidates(
    validation_sample,
    turn_options,
    reference_baseline,
)

# Attach hourly traffic features
validation_candidates = attach_hourly_features(
    validation_candidates,
    reference_hourly,
)

# Attach trajectory-history features
validation_candidates = attach_history_features(
    validation_candidates,
    validation_sample_history,
)

# Validate the resulting dataset
print("Validation movements:", len(validation_sample))
print("Candidate rows:", len(validation_candidates))

print(
    "Exactly one correct candidate per movement:",
    validation_candidates.groupby("movement_id")["chosen"]
    .sum().eq(1).all(),
)

print(
    "Missing model features:",
    validation_candidates[MODEL_FEATURES].isna().sum().sum(),
)

display(
    validation_candidates[
        [
            "movement_id",
            "incoming",
            "intersection",
            "outgoing",
            "chosen",
            "historical_probability",
            "adaptive_probability",
            "previous_road_1",
            "previous_road_2",
        ]
    ].head()
)

Validation movements: 10000
Candidate rows: 26170
Exactly one correct candidate per movement: True
Missing model features: 0


,movement_id,incoming,intersection,outgoing,chosen,historical_probability,adaptive_probability,previous_road_1,previous_road_2
0,0,1503,11345,3940,1,0.986460,0.979710,6892,6891
1,0,1503,11345,1499,0,0.013437,0.020278,6892,6891
2,0,1503,11345,11715,0,0.000103,0.000013,6892,6891
3,1,2992,2993,9907,1,0.388640,0.478225,11861,11860
4,1,2992,2993,11856,0,0.611360,0.521775,11861,11860


### Evaluation of LightGBM against historical baseline

In [19]:
# Historical baseline: probabilities already sum to 1 per movement
baseline_evaluation = validation_candidates[
    ["movement_id", "chosen", "historical_probability"]
].rename(
    columns={"historical_probability": "probability"}
)

baseline_log_loss = calculate_log_loss(baseline_evaluation)

# LightGBM: predict and normalize candidate scores
validation_scores = lightgbm_model.predict_proba(
    validation_candidates[MODEL_FEATURES]
)[:, 1]

model_evaluation = normalize_candidate_scores(
    validation_candidates,
    validation_scores,
)

model_log_loss = calculate_log_loss(model_evaluation)

# Relative improvement
improvement_pct = (
    (baseline_log_loss - model_log_loss)
    / baseline_log_loss
    * 100
)

print(f"Historical baseline log loss: {baseline_log_loss:.6f}")
print(f"LightGBM log loss:            {model_log_loss:.6f}")
print(f"Relative improvement:        {improvement_pct:.2f}%")

Historical baseline log loss: 0.322716
LightGBM log loss:            0.297542
Relative improvement:        7.80%


In [20]:
adaptive_evaluation = validation_candidates[
    ["movement_id", "chosen", "adaptive_probability"]
].rename(
    columns={"adaptive_probability": "probability"}
)

adaptive_log_loss = calculate_log_loss(adaptive_evaluation)

adaptive_improvement_pct = (
    (adaptive_log_loss - model_log_loss)
    / adaptive_log_loss
    * 100
)

print(f"Static baseline log loss:   {baseline_log_loss:.6f}")
print(f"Adaptive baseline log loss: {adaptive_log_loss:.6f}")
print(f"LightGBM log loss:          {model_log_loss:.6f}")
print(f"Improvement vs adaptive:    {adaptive_improvement_pct:.2f}%")

Static baseline log loss:   0.322716
Adaptive baseline log loss: 0.320283
LightGBM log loss:          0.297542
Improvement vs adaptive:    7.10%


### Independent test sample

In [21]:
# Calculate history on the complete held-out test set
test_history = build_trajectory_history(
    test_movements
)

# Sample movements while preserving their original indices
test_sample = test_movements.sample(
    n=10_000,
    random_state=RANDOM_STATE,
)

# Retrieve correctly aligned trajectory history
test_sample_history = test_history.loc[
    test_sample.index,
    ["previous_road_1", "previous_road_2", "trip_position"],
].reset_index(drop=True)

# Assign movement IDs after resetting the sample index
test_sample = test_sample.reset_index(drop=True)

test_sample_history["movement_id"] = np.arange(
    len(test_sample),
    dtype=np.int32,
)

print("Test movements:", len(test_sample))
print("History rows:", len(test_sample_history))
print("Missing history values:", test_sample_history.isna().sum().sum())

print(
    "Distinct test trips:",
    test_sample[["VehicleID", "TripID"]]
    .drop_duplicates().shape[0],
)

display(test_sample_history.head())

Test movements: 10000
History rows: 10000
Missing history values: 0
Distinct test trips: 7874


,previous_road_1,previous_road_2,trip_position,movement_id
0,7318,7325,12,0
1,1505,1295,37,1
2,6770,11871,7,2
3,-1,-1,0,3
4,9235,9246,36,4


### Test candidates

In [22]:
# Generate outgoing-road candidates
test_candidates = build_candidates(
    test_sample,
    turn_options,
    reference_baseline,
)

# Attach hourly and adaptive probability features
test_candidates = attach_hourly_features(
    test_candidates,
    reference_hourly,
)

# Attach correctly aligned trajectory history
test_candidates = attach_history_features(
    test_candidates,
    test_sample_history,
)

# Verify the resulting dataset
print("Test movements:", len(test_sample))
print("Candidate rows:", len(test_candidates))

print(
    "Exactly one correct candidate per movement:",
    test_candidates.groupby("movement_id")["chosen"]
    .sum().eq(1).all(),
)

print(
    "Missing model features:",
    test_candidates[MODEL_FEATURES].isna().sum().sum(),
)

display(
    test_candidates[
        [
            "movement_id",
            "incoming",
            "intersection",
            "outgoing",
            "chosen",
            "adaptive_probability",
            "previous_road_1",
            "previous_road_2",
        ]
    ].head()
)

Test movements: 10000
Candidate rows: 26171
Exactly one correct candidate per movement: True
Missing model features: 0


,movement_id,incoming,intersection,outgoing,chosen,adaptive_probability,previous_road_1,previous_road_2
0,0,11352,11863,6753,0,0.000007,7318,7325
1,0,11352,11863,6756,0,0.270087,7318,7325
2,0,11352,11863,6752,1,0.729905,7318,7325
3,1,3931,3932,3931,0,0.047501,1505,1295
4,1,3931,3932,11346,0,0.080741,1505,1295


### Evaluating LightGBM on the independent test set

In [23]:
# Evaluate adaptive hourly baseline
test_baseline_evaluation = test_candidates[
    ["movement_id", "chosen", "adaptive_probability"]
].rename(
    columns={"adaptive_probability": "probability"}
)

test_baseline_loss = calculate_log_loss(
    test_baseline_evaluation
)

# Predict candidate scores with LightGBM
test_scores = lightgbm_model.predict_proba(
    test_candidates[MODEL_FEATURES]
)[:, 1]

# Normalize scores into probabilities per movement
test_model_evaluation = normalize_candidate_scores(
    test_candidates,
    test_scores,
)

test_model_loss = calculate_log_loss(
    test_model_evaluation
)

# Calculate relative improvement
test_improvement_pct = (
    (test_baseline_loss - test_model_loss)
    / test_baseline_loss
    * 100
)

print(f"Adaptive baseline log loss: {test_baseline_loss:.6f}")
print(f"LightGBM log loss:          {test_model_loss:.6f}")
print(f"Relative improvement:      {test_improvement_pct:.2f}%")

Adaptive baseline log loss: 0.317135
LightGBM log loss:          0.293281
Relative improvement:      7.52%


### 95% bootstrap confidence interval

In [24]:
# Extract the probability assigned to the actual outgoing road
baseline_actual = test_baseline_evaluation.loc[
    test_baseline_evaluation["chosen"] == 1,
    ["movement_id", "probability"],
].rename(columns={"probability": "baseline_probability"})

model_actual = test_model_evaluation.loc[
    test_model_evaluation["chosen"] == 1,
    ["movement_id", "probability"],
].rename(columns={"probability": "model_probability"})

# Combine probabilities with trip identifiers
test_loss_data = (
    test_sample[["VehicleID", "TripID"]]
    .assign(movement_id=np.arange(len(test_sample)))
    .merge(baseline_actual, on="movement_id", validate="one_to_one")
    .merge(model_actual, on="movement_id", validate="one_to_one")
)

# Calculate per-movement log loss
test_loss_data["baseline_loss"] = -np.log(
    test_loss_data["baseline_probability"].clip(lower=1e-15)
)

test_loss_data["model_loss"] = -np.log(
    test_loss_data["model_probability"].clip(lower=1e-15)
)

# Bootstrap entire trips
ci_lower, ci_upper = bootstrap_trip_improvement(
    test_loss_data,
    n_bootstrap=1000,
    random_state=RANDOM_STATE,
)

print("Distinct trips:", test_loss_data[["VehicleID", "TripID"]].drop_duplicates().shape[0])
print(f"Observed improvement: {test_improvement_pct:.2f}%")
print(f"95% bootstrap CI: [{ci_lower:.2f}%, {ci_upper:.2f}%]")

Distinct trips: 7874
Observed improvement: 7.52%
95% bootstrap CI: [5.50%, 9.10%]


### Highway movements in the test sample

In [25]:
# Retain highway classifications from the held-out test split
test_highway_labels = highway_movements.loc[
    (highway_movements["split"] == "test")
    & highway_movements["highway_action"].isin(["continue", "exit"]),
    [
        "VehicleID",
        "TripID",
        "incoming",
        "intersection",
        "outgoing",
        "timestamp",
        "highway_action",
    ],
].drop_duplicates()

# Match classifications to sampled test movements
test_highway_sample = (
    test_sample
    .reset_index()
    .rename(columns={"index": "movement_id"})
    .merge(
        test_highway_labels,
        on=[
            "VehicleID",
            "TripID",
            "incoming",
            "intersection",
            "outgoing",
            "timestamp",
        ],
        how="inner",
        validate="many_to_one",
    )
)

print("Classified highway movements in test sample:", len(test_highway_sample))

print("\nHighway actions:")
print(test_highway_sample["highway_action"].value_counts())

print(
    "\nDistinct highway exit locations:",
    test_highway_sample["intersection"].nunique(),
)

print(
    "\nDuplicate movement IDs:",
    test_highway_sample["movement_id"].duplicated().sum(),
)

Classified highway movements in test sample: 23

Highway actions:
highway_action
exit        14
continue     9
Name: count, dtype: int64

Distinct highway exit locations: 8

Duplicate movement IDs: 0


### Full highway test subset

In [27]:
# Preserve original test indices for correct history alignment
highway_test_movements = (
    test_movements
    .reset_index(names="original_index")
    .merge(
        test_highway_labels,
        on=[
            "VehicleID",
            "TripID",
            "incoming",
            "intersection",
            "outgoing",
            "timestamp",
        ],
        how="inner",
        validate="many_to_one",
    )
)

highway_test_history = test_history.loc[
    highway_test_movements["original_index"],
    ["previous_road_1", "previous_road_2", "trip_position"],
].reset_index(drop=True)

highway_test_movements = highway_test_movements.drop(
    columns="original_index"
).reset_index(drop=True)

highway_test_history["movement_id"] = np.arange(
    len(highway_test_movements),
    dtype=np.int32,
)

print("Highway test movements:", len(highway_test_movements))
print("\nActions:")
print(highway_test_movements["highway_action"].value_counts())
print("\nMissing history features:", highway_test_history.isna().sum().sum())

Highway test movements: 5241

Actions:
highway_action
exit        3300
continue    1941
Name: count, dtype: int64

Missing history features: 0


### Highway test candidates

In [28]:
# Generate possible outgoing roads for highway movements
highway_test_candidates = build_candidates(
    highway_test_movements,
    turn_options,
    reference_baseline,
)

# Attach hourly and adaptive probability features
highway_test_candidates = attach_hourly_features(
    highway_test_candidates,
    reference_hourly,
)

# Attach trajectory-history features
highway_test_candidates = attach_history_features(
    highway_test_candidates,
    highway_test_history,
)

print("Highway test movements:", len(highway_test_movements))
print("Candidate rows:", len(highway_test_candidates))

print(
    "Exactly one correct candidate per movement:",
    highway_test_candidates.groupby("movement_id")["chosen"]
    .sum().eq(1).all(),
)

print(
    "Missing model features:",
    highway_test_candidates[MODEL_FEATURES].isna().sum().sum(),
)

print(
    "Unique evaluated intersections:",
    highway_test_movements["intersection"].nunique(),
)

Highway test movements: 5241
Candidate rows: 10940
Exactly one correct candidate per movement: True
Missing model features: 0
Unique evaluated intersections: 14


### Evaluate highway exit and continuation predictions

In [29]:
# Adaptive baseline probabilities
highway_baseline = highway_test_candidates[
    ["movement_id", "chosen", "adaptive_probability"]
].rename(columns={"adaptive_probability": "probability"})

# LightGBM predictions
highway_scores = lightgbm_model.predict_proba(
    highway_test_candidates[MODEL_FEATURES]
)[:, 1]

highway_model = normalize_candidate_scores(
    highway_test_candidates,
    highway_scores,
)

# Extract probabilities assigned to the actual outgoing road
highway_results = (
    highway_test_movements[["highway_action"]]
    .assign(movement_id=np.arange(len(highway_test_movements)))
    .merge(
        highway_baseline.loc[
            highway_baseline["chosen"] == 1,
            ["movement_id", "probability"],
        ].rename(columns={"probability": "baseline_probability"}),
        on="movement_id",
        validate="one_to_one",
    )
    .merge(
        highway_model.loc[
            highway_model["chosen"] == 1,
            ["movement_id", "probability"],
        ].rename(columns={"probability": "model_probability"}),
        on="movement_id",
        validate="one_to_one",
    )
)

# Per-movement log losses
highway_results["baseline_loss"] = -np.log(
    highway_results["baseline_probability"].clip(lower=1e-15)
)

highway_results["model_loss"] = -np.log(
    highway_results["model_probability"].clip(lower=1e-15)
)

# Compare performance by highway action
highway_performance = (
    highway_results.groupby("highway_action")
    .agg(
        movements=("movement_id", "count"),
        baseline_log_loss=("baseline_loss", "mean"),
        lightgbm_log_loss=("model_loss", "mean"),
    )
)

highway_performance["improvement_pct"] = (
    (
        highway_performance["baseline_log_loss"]
        - highway_performance["lightgbm_log_loss"]
    )
    / highway_performance["baseline_log_loss"]
    * 100
)

display(highway_performance.round(4))

,movements,baseline_log_loss,lightgbm_log_loss,improvement_pct
highway_action,,,,
continue,1941,0.1777,0.1640,7.6928
exit,3300,0.1740,0.1749,-0.4751


### Evaluate uncertainty in highway-specific performance

In [30]:
# Add trip identifiers to highway evaluation results
highway_loss_data = highway_results.merge(
    highway_test_movements[
        ["VehicleID", "TripID"]
    ].assign(
        movement_id=np.arange(len(highway_test_movements))
    ),
    on="movement_id",
    validate="one_to_one",
)

# Bootstrap relative improvement separately by action
for action in ["continue", "exit"]:
    subset = highway_loss_data.loc[
        highway_loss_data["highway_action"] == action
    ]

    lower, upper = bootstrap_trip_improvement(
        subset,
        n_bootstrap=1000,
        random_state=RANDOM_STATE,
    )

    observed = highway_performance.loc[
        action, "improvement_pct"
    ]

    print(f"{action.capitalize()}:")
    print(f"  Movements: {len(subset)}")
    print(f"  Distinct trips: {subset[['VehicleID', 'TripID']].drop_duplicates().shape[0]}")
    print(f"  Improvement: {observed:.2f}%")
    print(f"  95% bootstrap CI: [{lower:.2f}%, {upper:.2f}%]")
    print()

Continue:
  Movements: 1941
  Distinct trips: 1022
  Improvement: 7.69%
  95% bootstrap CI: [7.04%, 8.43%]

Exit:
  Movements: 3300
  Distinct trips: 2062
  Improvement: -0.48%
  95% bootstrap CI: [-2.63%, 1.81%]



In [36]:
import pickle
import joblib

CACHE_DIR = PROJECT_ROOT / "data" / "processed" / "shenzhen"
MODEL_DIR = PROJECT_ROOT / "models"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

datasets_to_save = {
    "model_movements": model_movements,
    "reference_movements": reference_movements,
    "model_train_movements": model_train_movements,
    "train_movements": train_movements,
    "validation_movements": validation_movements,
    "test_movements": test_movements,
    "turn_options": turn_options,
    "reference_baseline": reference_baseline,
    "reference_hourly": reference_hourly,
    "highway_movements": highway_movements,
}

# Save each DataFrame
for name, df in datasets_to_save.items():
    path = CACHE_DIR / f"{name}.pkl"

    with open(path, "wb") as f:
        pickle.dump(df, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Saved {name}: {len(df):,} rows")

# Save the trained LightGBM model
model_path = MODEL_DIR / "shenzhen_lightgbm.joblib"
joblib.dump(lightgbm_model, model_path)

print(f"\nModel saved: {model_path}")
print("\nAll files saved successfully!")

Saved model_movements: 14,313,159 rows
Saved reference_movements: 7,015,331 rows
Saved model_train_movements: 3,035,193 rows
Saved train_movements: 10,050,524 rows
Saved validation_movements: 2,095,554 rows
Saved test_movements: 2,167,081 rows
Saved turn_options: 72,495 rows
Saved reference_baseline: 72,495 rows
Saved reference_hourly: 724,950 rows
Saved highway_movements: 39,282 rows

Model saved: C:\Users\EAG28\OneDrive\Документы\2026W\Project\ev-traffic-charging-prediction\models\shenzhen_lightgbm.joblib

All files saved successfully!


In [ ]:
import pickle
import joblib

# Verify that all saved files exist
for name in datasets_to_save:
    path = CACHE_DIR / f"{name}.pkl"
    assert path.exists() and path.stat().st_size > 0, name

# Test loading the probability table
with open(CACHE_DIR / "reference_baseline.pkl", "rb") as f:
    restored_baseline = pickle.load(f)

# Test loading the trained model
restored_model = joblib.load(
    MODEL_DIR / "shenzhen_lightgbm.joblib"
)

print("All dataset files exist: True")
print("Restored baseline rows:", len(restored_baseline))
print("Restored model features:", restored_model.n_features_in_)
print("Ready for next session!")

All dataset files exist: True
Restored baseline rows: 72495
Restored model features: 14
Ready for next session!


## Results and Discussion

### General Next-Road Prediction

The LightGBM model outperformed the adaptive hourly turning-probability
baseline on both validation and independent test data.

| Metric | Validation | Test |
|---|---:|---:|
| Adaptive baseline log loss | 0.320283 | 0.317135 |
| LightGBM log loss | 0.297542 | 0.293281 |
| Relative improvement | 7.10% | 7.52% |

A trip-level bootstrap with 1,000 resamples produced a 95% confidence
interval of [5.50%, 9.10%] for the relative test improvement.

These results suggest that combining historical traffic patterns with
recent trajectory information improves next-road probability estimation.

### Highway-Specific Evaluation

We identified 196 potential highway exit locations from the road network.
However, classified trajectory movements were observed at only 14
locations, including 9 with observed exit movements.

The highway evaluation used 5,241 movements from the independent test set.

| Highway action | Movements | Relative improvement | 95% bootstrap CI |
|---|---:|---:|---:|
| Continue | 1,941 | +7.69% | [+7.04%, +8.43%] |
| Exit | 3,300 | -0.48% | [-2.63%, +1.81%] |

LightGBM substantially improved next-road prediction for highway
continuations. However, no clear improvement was established for
highway exits.

### Limitations

- Highway observations were concentrated at a small number of
  intersections, limiting geographic generalizability.
- The model was trained for general next-road prediction rather than
  specifically for highway exit decisions.
- The highway evaluation measures the probability assigned to the
  actual outgoing road, not binary exit-versus-continue classification.